# Text Generation Architecture Comparison
### LSTM vs GRU vs FNet - a controlled, character-level comparison

This notebook trains three different sequence architectures on the **same data**, with the **same task**
(next-character prediction) and the **same training loop**, so that differences in results can be attributed
to the architecture rather than to inconsistent setup.

**Models compared:**
1. **LSTM** - recurrent, gated, sequential processing
2. **GRU** - recurrent, simplified gating, fewer parameters
3. **FNet (decoder-only)** - replaces self-attention with a Fourier Transform for token mixing, processed in parallel rather than sequentially

**Dataset:** *Alice's Adventures in Wonderland* (Lewis Carroll, public domain, via Project Gutenberg)

**What we'll measure for each model:**
- Parameter count
- Training time per epoch
- Training loss / perplexity curve
- Generated text samples from the same seed
- Qualitative comparison (coherence, repetition, artifacts)

**Roadmap for this notebook:**
- Part 1: Shared setup & data pipeline
- Part 2: LSTM model
- Part 3: GRU model
- Part 4: FNet model (decoder-only)
- Part 5: Shared training harness
- Part 6: Train all three
- Part 7: Comparison & analysis


## Part 1: Shared Setup & Data Pipeline

Using the **same** data pipeline for all three models is the whole point of a fair comparison - same vocab,
same sequence length, same train/val split, same batching.


In [ ]:
%pip install torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import time
import random
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
import os

DATA_PATH = 'alice.txt'
URL = 'https://raw.githubusercontent.com/GITenberg/Alice-s-Adventures-in-Wonderland_11/master/11-0.txt'

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)

with open(DATA_PATH, 'r', encoding='utf-8-sig') as f:
    raw_text = f.read()

print(f'Raw length: {len(raw_text):,} characters')
print(raw_text[:300])

In [ ]:
#Clean the text

def clean_gutenberg_text(text):
    # Strip Gutenberg header/footer boilerplate
    start_marker = '*** START OF'
    end_marker = '*** END OF'
    start_idx = text.find(start_marker)
    end_idx = text.find(end_marker)
    if start_idx != -1:
        start_idx = text.find('\n', start_idx) + 1
        text = text[start_idx:]
    if end_idx != -1:
        text = text[:text.find(end_marker)]

    # Normalize whitespace and curly quotes, keep punctuation
    text = text.replace('\r\n', '\n')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.lower()
    text = '\n'.join(line.strip() for line in text.split('\n'))
    while '\n\n\n' in text:
        text = text.replace('\n\n\n', '\n\n')

    return text.strip()

text = clean_gutenberg_text(raw_text)
print(f'Cleaned length: {len(text):,} characters')
print(text[:300])


In [ ]:
#Build the character vocabulary

vocab = sorted(set(text))
vocab_size = len(vocab)
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}

print(f'Vocabulary size: {vocab_size}')
print(f'Vocabulary: {"".join(vocab)!r}')

text_as_int = np.array([char_to_idx[c] for c in text], dtype=np.int64)
print(f'Encoded text shape: {text_as_int.shape}')


In [ ]:
#Train / validation split

split_idx = int(len(text_as_int) * 0.9)
train_data = text_as_int[:split_idx]
val_data = text_as_int[split_idx:]

print(f'Train characters: {len(train_data):,}')
print(f'Val characters:   {len(val_data):,}')

In [ ]:
#Sequence dataset

from torch.utils.data import Dataset, DataLoader

SEQ_LENGTH = 100

class CharDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_length]
        y = self.data[idx + 1: idx + self.seq_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_dataset = CharDataset(train_data, SEQ_LENGTH)
val_dataset = CharDataset(val_data, SEQ_LENGTH)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

# sanity check
xb, yb = next(iter(train_loader))
print(f'Batch input shape:  {xb.shape}')
print(f'Batch target shape: {yb.shape}')
print('Sample input :', ''.join(idx_to_char[i.item()] for i in xb[0][:50]))
print('Sample target:', ''.join(idx_to_char[i.item()] for i in yb[0][:50]))


In [ ]:
#LSTM model


class LSTMTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len) -> embedded: (batch, seq_len, embed_dim)
        embedded = self.embedding(x)
        # output: (batch, seq_len, hidden_dim)
        output, hidden = self.lstm(embedded, hidden)
        # logits: (batch, seq_len, vocab_size)
        logits = self.fc(output)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        return (h0, c0)

In [ ]:
#Sanity Check

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Shared hyperparameters — GRU and FNet will use the same embed_dim/hidden_dim
# so parameter counts and capacity are comparable across architectures.
EMBED_DIM = 64
HIDDEN_DIM = 128

lstm_model = LSTMTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
print(lstm_model)
print(f'\nTotal trainable parameters: {count_parameters(lstm_model):,}')

# forward pass sanity check
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
logits, hidden = lstm_model(xb)

print(f'\nInput shape:  {xb.shape}')
print(f'Logits shape: {logits.shape}  (batch, seq_len, vocab_size)')
assert logits.shape == (xb.shape[0], xb.shape[1], vocab_size), 'Shape mismatch!'

loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
print(f'Loss on one untrained batch: {loss.item():.4f}')
print(f'Reference (random guessing) loss: {np.log(vocab_size):.4f}')


In [ ]:
#GRU model

class GRUTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: (batch, seq_len) -> embedded: (batch, seq_len, embed_dim)
        embedded = self.embedding(x)
        # output: (batch, seq_len, hidden_dim); hidden: (num_layers, batch, hidden_dim)
        output, hidden = self.gru(embedded, hidden)
        # logits: (batch, seq_len, vocab_size)
        logits = self.fc(output)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        # GRU only has one state tensor (no separate cell state like LSTM)
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)

In [ ]:
#Sanity Check

gru_model = GRUTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
print(gru_model)
print(f'\nTotal trainable parameters: {count_parameters(gru_model):,}')

lstm_params = count_parameters(lstm_model)
gru_params = count_parameters(gru_model)
print(f'\nLSTM params: {lstm_params:,}')
print(f'GRU params:  {gru_params:,}')
print(f'Difference:  {lstm_params - gru_params:,} fewer params in GRU '
      f'({(1 - gru_params/lstm_params)*100:.1f}% smaller)')

# forward pass sanity check
logits, hidden = gru_model(xb)

print(f'\nInput shape:  {xb.shape}')
print(f'Logits shape: {logits.shape}  (batch, seq_len, vocab_size)')
assert logits.shape == (xb.shape[0], xb.shape[1], vocab_size), 'Shape mismatch!'

loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
print(f'Loss on one untrained batch: {loss.item():.4f}')
print(f'Reference (random guessing) loss: {np.log(vocab_size):.4f}')

In [ ]:
#FNet
#Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:x.size(1)].unsqueeze(0)


In [ ]:
#Causal Fourier mixing block
class CausalFNetBlock(nn.Module):
    def __init__(self, embed_dim, dense_dim, window_size=32):
        super().__init__()
        self.window_size = window_size
        self.dense_proj = nn.Sequential(
            nn.Linear(embed_dim, dense_dim),
            nn.ReLU(),
            nn.Linear(dense_dim, embed_dim),
        )
        self.layernorm_1 = nn.LayerNorm(embed_dim)
        self.layernorm_2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        batch, seq_len, embed_dim = x.shape
        W = self.window_size

        # left-pad so every position (including the first) has a full causal window
        x_padded = F.pad(x, (0, 0, W - 1, 0))  # pad the sequence dim on the left only

        # unfold into overlapping causal windows: (batch, seq_len, embed_dim, W) -> (batch, seq_len, W, embed_dim)
        windows = x_padded.unfold(dimension=1, size=W, step=1).permute(0, 1, 3, 2)

        # 2D FFT over (window position, embedding channel), keep the real part -
        # this is the actual FNet mechanic, just scoped to a causal window
        fft_real = torch.fft.fft2(windows).real  # (batch, seq_len, W, embed_dim)

        # take the representation at the most recent (current) position in each window
        mixed = fft_real[:, :, -1, :]  # (batch, seq_len, embed_dim)

        proj_input = self.layernorm_1(x + mixed)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)


In [ ]:
#Full model
class FNetTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, dense_dim=128, num_blocks=2,
                 window_size=32, max_seq_length=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, max_seq_length)
        self.blocks = nn.ModuleList([
            CausalFNetBlock(embed_dim, dense_dim, window_size) for _ in range(num_blocks)
        ])
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, hidden=None):
        # hidden is accepted (and ignored) so this matches the LSTM/GRU call signature
        embedded = self.embedding(x)
        h = self.pos_encoding(embedded)
        for block in self.blocks:
            h = block(h)
        logits = self.fc(h)
        return logits, None

    def init_hidden(self, batch_size, device):
        # stateless model - nothing to carry between calls
        return None

In [ ]:
#Sanity Check
fnet_model = FNetTextGenerator(
    vocab_size, embed_dim=EMBED_DIM, dense_dim=HIDDEN_DIM,
    num_blocks=2, window_size=32, max_seq_length=SEQ_LENGTH
).to(device)

print(fnet_model)
print(f'\nTotal trainable parameters: {count_parameters(fnet_model):,}')
print(f'(for reference — LSTM: {count_parameters(lstm_model):,}, GRU: {count_parameters(gru_model):,})')

t0 = time.time()
logits, _ = fnet_model(xb)
elapsed = time.time() - t0

print(f'\nInput shape:  {xb.shape}')
print(f'Logits shape: {logits.shape}  (batch, seq_len, vocab_size)')
assert logits.shape == (xb.shape[0], xb.shape[1], vocab_size), 'Shape mismatch!'
print(f'Forward pass time: {elapsed*1000:.1f} ms')

loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
print(f'Loss on one untrained batch: {loss.item():.4f}')
print(f'Reference (random guessing) loss: {np.log(vocab_size):.4f}')

In [ ]:
#Verifying causality directly
fnet_model.eval()
with torch.no_grad():
    test_seq = xb[:1].clone()  # single sequence, shape (1, seq_len)

    logits_a, _ = fnet_model(test_seq)

    test_seq_modified = test_seq.clone()
    check_pos = 10  # we'll verify position `check_pos` is unaffected by changes after it
    test_seq_modified[0, check_pos + 1:] = torch.randint(
        0, vocab_size, (test_seq.shape[1] - check_pos - 1,), device=device
    )

    logits_b, _ = fnet_model(test_seq_modified)

    before = logits_a[0, :check_pos + 1]
    after = logits_b[0, :check_pos + 1]
    max_diff = (before - after).abs().max().item()

    print(f'Max logit difference at positions 0..{check_pos} after changing future tokens: {max_diff:.2e}')
    assert max_diff < 1e-4, 'Causality violated - future tokens are leaking into past predictions!'
    print('Causal - no future leakage detected.')

fnet_model.train()

In [ ]:
#Shared Training Harness
def train_epoch(model, loader, optimizer, device, max_batches=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for i, (xb, yb) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for i, (xb, yb) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        xb, yb = xb.to(device), yb.to(device)

        logits, _ = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))

        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / n_batches
    perplexity = float(np.exp(avg_loss))
    return avg_loss, perplexity


In [ ]:
#Full training orchestration
def train_model(model, model_name, train_loader, val_loader, epochs, lr, device, max_batches=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'model_name': model_name,
        'param_count': count_parameters(model),
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'epoch_times': [],
    }

    print(f'Training {model_name} ({history["param_count"]:,} parameters)')
    total_start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, device, max_batches)
        val_loss, val_ppl = evaluate(model, val_loader, device, max_batches)

        epoch_time = time.time() - epoch_start

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_perplexity'].append(val_ppl)
        history['epoch_times'].append(epoch_time)

        print(f'  Epoch {epoch+1}/{epochs} | train_loss: {train_loss:.4f} | '
              f'val_loss: {val_loss:.4f} | val_ppl: {val_ppl:.2f} | time: {epoch_time:.1f}s')

    total_time = time.time() - total_start
    history['total_train_time'] = total_time
    print(f'  Total training time: {total_time:.1f}s\n')

    return history

In [ ]:
#Shared text generation function
@torch.no_grad()
def generate_text(model, seed_text, length=200, temperature=0.7, device=device, max_context=SEQ_LENGTH):
    model.eval()
    generated = seed_text.lower()

    for _ in range(length):
        context = generated[-max_context:]
        input_ids = torch.tensor(
            [[char_to_idx.get(c, 0) for c in context]], dtype=torch.long, device=device
        )

        logits, _ = model(input_ids)
        next_logits = logits[0, -1, :] / temperature
        probs = F.softmax(next_logits, dim=0)
        next_idx = torch.multinomial(probs, num_samples=1).item()

        generated += idx_to_char[next_idx]

    model.train()
    return generated


In [ ]:
#Smoke test
# quick smoke test: a few batches only, just to confirm the harness runs end-to-end without errors
test_model = LSTMTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)
smoke_history = train_model(
    test_model, 'LSTM (smoke test)', train_loader, val_loader,
    epochs=1, lr=0.002, device=device, max_batches=5
)

sample = generate_text(test_model, seed_text='alice was ', length=80, max_context=SEQ_LENGTH)
print('Sample generation (from a barely-trained model, expect gibberish):')
print(sample)


In [ ]:
#Train All Three Models
import matplotlib.pyplot as plt

# --- Training configuration ---
EPOCHS = 10
MAX_BATCHES_PER_EPOCH = 250   # set to None to use the full 2,016 batches/epoch
MAX_VAL_BATCHES = 50          # set to None to use the full validation set
LEARNING_RATE = 0.002

print(f'Config: {EPOCHS} epochs, '
      f'{MAX_BATCHES_PER_EPOCH or len(train_loader)} train batches/epoch, '
      f'{MAX_VAL_BATCHES or len(val_loader)} val batches/epoch')


In [ ]:
#Fresh model instances
torch.manual_seed(SEED)  # same initialization conditions across the notebook re-run

models_to_train = {
    'LSTM': LSTMTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device),
    'GRU': GRUTextGenerator(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device),
    'FNet': FNetTextGenerator(
        vocab_size, embed_dim=EMBED_DIM, dense_dim=HIDDEN_DIM,
        num_blocks=1, window_size=16, max_seq_length=SEQ_LENGTH
    ).to(device),
}

for name, m in models_to_train.items():
    print(f'{name}: {count_parameters(m):,} parameters')

In [ ]:
#Train each model
histories = {}

for name, model in models_to_train.items():
    histories[name] = train_model(
        model, name, train_loader, val_loader,
        epochs=EPOCHS, lr=LEARNING_RATE, device=device,
        max_batches=MAX_BATCHES_PER_EPOCH,
    )


In [ ]:
#Loss curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'LSTM': '#2563eb', 'GRU': '#16a34a', 'FNet': '#dc2626'}

for name, h in histories.items():
    axes[0].plot(range(1, EPOCHS + 1), h['train_loss'], label=name, color=colors[name], marker='o', markersize=3)
    axes[1].plot(range(1, EPOCHS + 1), h['val_loss'], label=name, color=colors[name], marker='o', markersize=3)

axes[0].set_title('Training Loss')
axes[1].set_title('Validation Loss')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cross-entropy loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
#Generated samples
SEED_TEXT = "alice was "

for name, model in models_to_train.items():
    print(f'--- {name} ---')
    sample = generate_text(model, SEED_TEXT, length=250, temperature=0.7)
    print(sample)
    print()

In [ ]:
#Comparison & Analysis
summary_rows = []
for name, h in histories.items():
    summary_rows.append({
        'Model': name,
        'Parameters': h['param_count'],
        'Final Train Loss': h['train_loss'][-1],
        'Final Val Loss': h['val_loss'][-1],
        'Final Val Perplexity': h['val_perplexity'][-1],
        'Avg Time/Epoch (s)': np.mean(h['epoch_times']),
        'Total Train Time (s)': h['total_train_time'],
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.round(3)
summary_df

In [ ]:
#Efficiency: parameters vs. quality
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

names = list(histories.keys())
colors_list = [colors[n] for n in names]

axes[0].bar(names, [histories[n]['param_count'] for n in names], color=colors_list)
axes[0].set_title('Parameter Count')
axes[0].set_ylabel('Parameters')

axes[1].bar(names, [histories[n]['val_perplexity'][-1] for n in names], color=colors_list)
axes[1].set_title('Final Validation Perplexity\n(lower is better)')
axes[1].set_ylabel('Perplexity')

axes[2].bar(names, [histories[n]['total_train_time'] for n in names], color=colors_list)
axes[2].set_title('Total Training Time')
axes[2].set_ylabel('Seconds')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_summary.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
param_efficiency = summary_df['Final Val Perplexity'] / (summary_df['Parameters'] / 1000)
fastest_model = summary_df['Total Train Time (s)'].idxmin()
lowest_perplexity_model = summary_df['Final Val Perplexity'].idxmin()
most_efficient_model = param_efficiency.idxmin()

print("Suggested talking points based on this run:\n")
print(f"- Fastest to train: {fastest_model} "
      f"({summary_df.loc[fastest_model, 'Total Train Time (s)']:.1f}s total)")
print(f"- Lowest validation perplexity: {lowest_perplexity_model} "
      f"({summary_df.loc[lowest_perplexity_model, 'Final Val Perplexity']:.2f})")
print(f"- Best perplexity per 1K parameters: {most_efficient_model} "
      f"({param_efficiency[most_efficient_model]:.3f})")
